# Aether-Calc: Temporal Hyper-Relaxation Optimization

This notebook demonstrates the Aether-Calculus framework which implements:
- **Temporal hyper-relaxation** using Shannon entropy
- **Volatility surface modeling** for financial ML
- **Granular arithmetic** for uncertainty propagation

## Mathematical Foundation

Aether-Calc uses entropy-based relaxation:
```
θ_{t+1} = θ_t - α_t ∇L(θ_t) + β_t H(θ_t)
```

Where H is the Shannon entropy term that prevents premature convergence.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import sys
sys.path.insert(0, '/home/runner/workspace/Advanced-Research/Ideas/projects/aether_calc/src')

from aether_calc import (
    AetherOptimizer,
    AetherConfig,
    VolatilitySurface,
    GranularArithmeticUnit,
)

print("Aether-Calc imported successfully!")

## 1. Aether Optimizer

The AetherOptimizer implements temporal hyper-relaxation with entropy-based exploration.

In [ ]:
# Configure the Aether optimizer
config = AetherConfig(
    learning_rate=0.01,
    entropy_coefficient=0.1,
    temporal_decay=0.95,
    hyper_relaxation=0.5,
)

# Create a simple model
model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 2),
)

# Initialize optimizer
optimizer = AetherOptimizer(model.parameters(), config)

print(f"Aether Optimizer configured:")
print(f"  - Learning rate: {config.learning_rate}")
print(f"  - Entropy coefficient: {config.entropy_coefficient}")
print(f"  - Temporal decay: {config.temporal_decay}")
print(f"  - Hyper-relaxation: {config.hyper_relaxation}")

## 2. Training Loop with Aether Optimizer

In [ ]:
# Generate synthetic data
torch.manual_seed(42)
X = torch.randn(200, 10)
y = (X @ torch.randn(10, 2) > 0).float()

# Training loop
losses = []
entropy_terms = []

model.train()
for epoch in range(100):
    optimizer.zero_grad()
    
    output = model(X)
    loss = nn.functional.binary_cross_entropy_with_logits(output, y)
    
    loss.backward()
    entropy_term = optimizer.step()
    
    losses.append(loss.item())
    entropy_terms.append(entropy_term)
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}, Entropy = {entropy_term:.4f}")

In [ ]:
# Plot training dynamics
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(losses)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(entropy_terms)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Entropy Term')
axes[1].set_title('Entropy Regularization')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Volatility Surface Modeling

The VolatilitySurface class models implied volatility across strike prices and maturities.

In [ ]:
# Create volatility surface
strikes = np.linspace(80, 120, 20)
maturities = np.linspace(0.1, 2.0, 20)

surface = VolatilitySurface(strikes, maturities)
vol_matrix = surface.compute_surface()

print(f"Volatility surface shape: {vol_matrix.shape}")
print(f"ATM volatility: {surface.get_atm_volatility(1.0):.4f}")

In [ ]:
# Visualize volatility surface
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

S, T = np.meshgrid(strikes, maturities)
ax.plot_surface(S, T, vol_matrix, cmap='viridis', alpha=0.8)

ax.set_xlabel('Strike Price')
ax.set_ylabel('Maturity (Years)')
ax.set_zlabel('Implied Volatility')
ax.set_title('Volatility Surface')

plt.tight_layout()
plt.show()

## 4. Granular Arithmetic Unit

The GranularArithmeticUnit performs arithmetic with uncertainty bounds.

In [ ]:
# Create granular arithmetic unit
gau = GranularArithmeticUnit()

# Perform operations with uncertainty
a = gau.create_granular(10.0, 0.5)  # value=10, uncertainty=0.5
b = gau.create_granular(5.0, 0.3)

c_add = gau.add(a, b)
c_mul = gau.multiply(a, b)

print(f"a = {a}")
print(f"b = {b}")
print(f"a + b = {c_add}")
print(f"a * b = {c_mul}")

In [ ]:
# Demonstrate uncertainty propagation
x_vals = np.linspace(0, 10, 50)
results = []

for x in x_vals:
    gx = gau.create_granular(x, x * 0.1)  # 10% uncertainty
    result = gau.exp(gx)  # e^x with uncertainty
    results.append((result.value, result.uncertainty))

values, uncertainties = zip(*results)

plt.figure(figsize=(10, 5))
plt.plot(x_vals, values, 'b-', label='exp(x)')
plt.fill_between(x_vals, 
                 np.array(values) - np.array(uncertainties),
                 np.array(values) + np.array(uncertainties),
                 alpha=0.3, label='Uncertainty bounds')
plt.xlabel('x')
plt.ylabel('exp(x)')
plt.title('Uncertainty Propagation in Granular Arithmetic')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Summary

Aether-Calc provides:
1. **AetherOptimizer**: Temporal hyper-relaxation with entropy-based exploration
2. **VolatilitySurface**: Financial volatility modeling
3. **GranularArithmeticUnit**: Uncertainty quantification through arithmetic operations